[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Tamaño del efecto.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo III: *Inferencia estadística*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se reportará el tamaño del efecto (d de Cohen, V de Cramér y η²) junto con el valor *p* y se integrarán las pruebas de hipótesis vistas en el módulo.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* 🟢 **Ejercicio básico:** lo que todos deben dominar.
* 🔴 **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencias.

py311 no cubre inferencia, así que la referencia de la API es la documentación oficial: [*SciPy* · `scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html) (`ttest_ind`, `mannwhitneyu`, `permutation_test`, `chi2_contingency`, `f_oneway`, `levene`), [*statsmodels* · pruebas de proporciones](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportions_ztest.html), [*statsmodels* · `pairwise_tukeyhsd`](https://www.statsmodels.org/stable/generated/statsmodels.stats.multicomp.pairwise_tukeyhsd.html) y [*statsmodels* · `anova_oneway`](https://www.statsmodels.org/stable/generated/statsmodels.stats.oneway.anova_oneway.html).

### Preparación del entorno.

* La siguiente celda importará las bibliotecas y las funciones de *SciPy* y *statsmodels* que se usarán, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.proportion import proportions_ztest

* La siguiente celda leerá `penguins_limpio.csv` (o lo regenerará si no existe), `titanic.csv` y `tips.csv`, y creará `tip_pct`.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio)
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
propinas = pd.read_csv(os.path.join(DATA, "tips.csv")).assign(tip_pct=lambda d: 100 * d["tip"] / d["total_bill"])
pinguinos.shape, titanic.shape, propinas.shape

### Objetos de los *notebooks* anteriores.

Este *notebook* es independiente: la siguiente celda vuelve a crear lo que se necesita de los *notebooks* anteriores.

* La siguiente celda definirá `d_cohen` y `eta_cuadrada`, presentadas en `20_pruebas_para_dos_grupos.ipynb` y `21_proporciones_y_mas_de_dos_grupos.ipynb`, y la tabla `tabla_clase` de `class` contra `survived`.

In [ ]:
def d_cohen(a, b):
    a, b = np.asarray(a), np.asarray(b)
    varianza_combinada = ((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2)
    return (a.mean() - b.mean()) / np.sqrt(varianza_combinada)


def eta_cuadrada(valores, grupos):
    datos = pd.DataFrame({"valor": valores, "grupo": grupos}).dropna()
    media_global = datos["valor"].mean()
    por_grupo = datos.groupby("grupo")["valor"].agg(["count", "mean"])
    suma_entre = (por_grupo["count"] * (por_grupo["mean"] - media_global) ** 2).sum()
    suma_total = ((datos["valor"] - media_global) ** 2).sum()
    return suma_entre / suma_total


tabla_clase = pd.crosstab(titanic["class"], titanic["survived"])
tabla_clase

---
## 1. Tamaño del efecto y ejercicio de la sesión (20 min).

| Situación | Prueba | Tamaño del efecto |
|---|---|---|
| Dos medias | Welch | d de Cohen (y la diferencia con su IC, en unidades originales) |
| Dos distribuciones asimétricas | Mann-Whitney o permutación de medianas | Diferencia de medianas con IC *bootstrap* |
| Dos proporciones | *z* de proporciones | Diferencia de proporciones o riesgo relativo |
| Dos categóricas | Chi cuadrada | V de Cramér |
| Tres o más medias | ANOVA (Welch si las varianzas difieren) + Tukey | η² |

**Plantilla para redactar:** "En estos datos, [grupo A] tuvo [medida] [mayor/menor] que [grupo B] ([valor A] contra [valor B]; diferencia [d], IC 95% [a, b]; [prueba], *p* [valor]; [tamaño del efecto] = [valor], un efecto [magnitud])." Sin verbos causales ("provoca", "hace que") cuando los datos son observacionales.

**¿Por qué esta sección?** El producto final de una prueba no es el valor *p*, sino una **frase** que una persona sin formación estadística pueda leer y que no exagere. La tabla resume qué tamaño del efecto acompaña a cada prueba, y la plantilla es la que se usará en el informe del proyecto (sesiones 7 y 8).

**💡 Ideas clave de la sección 1:** cada prueba tiene su tamaño del efecto; se reporta magnitud, IC, prueba y valor *p*, en lenguaje descriptivo.

### 🟢 Ejercicio 1a · Masa por sexo en Adelie.

**🎯 Objetivo:** redactar un resultado con la plantilla. **Qué se espera aprender:** a comunicar magnitud, incertidumbre y prueba en una sola frase, sin lenguaje causal.

Con los resultados de la sección 1 de `20_pruebas_para_dos_grupos.ipynb`, redacta la conclusión de la comparación de masa entre machos y hembras Adelie en una o dos frases, con la plantilla.

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 1b · Aleta por especie.

**🎯 Objetivo:** recorrer el flujo completo para más de dos grupos (Levene → ANOVA → Tukey → η²). **Qué se espera aprender:** a decidir cada paso según el anterior y a redactar la conclusión con el tamaño del efecto.

Compara `flipper_length_mm` entre las tres especies: revisa Levene, aplica el ANOVA (y el de Welch), calcula η² y, si corresponde, Tukey. Redacta la conclusión.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 1c · Clase y supervivencia.

**🎯 Objetivo:** cuantificar una asociación entre categóricas. **Qué se espera aprender:** a acompañar la chi cuadrada con la V de Cramér y las tasas por grupo.

Con la chi cuadrada de la sección 1 de `21_proporciones_y_mas_de_dos_grupos.ipynb`, calcula la V de Cramér de `class` contra `survived` y redacta la conclusión sin lenguaje causal injustificado.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 1 · Diagnóstico gráfico de supuestos.

**🎯 Objetivo:** revisar los supuestos del ANOVA con gráficas. **Qué se espera aprender:** a juzgar si una desviación de los supuestos puede cambiar la conclusión.

Para el ejercicio 1b, dibuja un gráfico Q-Q de `flipper_length_mm` de cada especie (`stats.probplot(..., plot=ax)`) y un diagrama de caja por especie. ¿Las conclusiones del ANOVA dependen de algún supuesto dudoso?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## ✅ Cierre.

**Hoy vimos:**
* La lógica del contraste, qué **no** significa un valor *p*, la potencia y por qué las comparaciones múltiples producen falsos hallazgos.
* Dos grupos: Welch por defecto (Student falla con varianzas y tamaños distintos), Mann-Whitney y permutación.
* Proporciones (*z*) y categóricas (chi cuadrada, revisando las frecuencias esperadas).
* Más de dos grupos: Levene, ANOVA clásico y de Welch, Tukey solo si el ANOVA rechaza.
* Tamaño del efecto: d de Cohen, V de Cramér y η²; con *n* enorme, significativo no implica importante.

**Práctica (sin calificación, en Classroom):** completar los tres incisos del ejercicio 1 y aplicar al menos una prueba con su tamaño del efecto al *dataset* del proyecto.

**Siguiente *notebook*:** `23_eda_completo_con_mpg.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>